# OpenFace Face Recognition
This notebook demonstrates face recognition using OpenFace, a general-purpose face recognition library with deep neural networks.

## 1. Install Dependencies and Import Libraries

In [67]:
# Upgrade build tooling
!pip install --upgrade pip setuptools wheel

# Pin versions known to work together on py3.12
!pip install --no-cache-dir \
  "numpy==1.26.4" \
  "protobuf>=4.25.3,<5" \
  "mediapipe==0.10.21" \
  "opencv-python==4.11.0.86" \
  "matplotlib==3.10.6" \
  "pillow==11.3.0" \
  "scikit-learn==1.6.1"

In [2]:
import sys, importlib

# show versions
import numpy as np, cv2, mediapipe as mp, matplotlib, sklearn, PIL
print("numpy:", np.__version__)
print("opencv:", cv2.__version__)
print("mediapipe:", getattr(mp, "__version__", "unknown"))
print("matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
print("pillow:", PIL.__version__)

# quick mediapipe sanity check
from mediapipe import solutions as mp_solutions
with mp_solutions.hands.Hands(static_image_mode=True) as hands:
    print("MediaPipe Hands initialized ✓")


numpy: 1.26.4
opencv: 4.11.0
mediapipe: 0.10.21
matplotlib: 3.10.6
scikit-learn: 1.6.1
pillow: 11.3.0
MediaPipe Hands initialized ✓


## 2. Define Helper Functions

## 2a. Optional Backends

In [16]:
# Initialize MediaPipe face detection components
print("Initializing MediaPipe face detection...")

# Flags to keep track of optional backends
FACE_RECOGNITION_AVAILABLE = False
OPENFACE_AVAILABLE = False
openface_net = None
openface_aligner = None

from pathlib import Path

OPENFACE_MODEL_PATH = Path("models/openface/nn4.small2.v1.t7")
OPENFACE_LANDMARK_PATH = Path("models/dlib/shape_predictor_68_face_landmarks.dat")
OPENFACE_MODEL_URL = "https://github.com/cmusatyalab/openface/raw/master/models/openface/nn4.small2.v1.t7"
OPENFACE_LANDMARK_URL = "https://github.com/cmusatyalab/openface/raw/master/models/dlib/shape_predictor_68_face_landmarks.dat.bz2"

try:
    # Import required torch for embeddings
    import torch
    from torch.nn.functional import cosine_similarity
    print("✓ PyTorch loaded")
    
    # Initialize MediaPipe face detection and mesh
    mp_face_detection = mp.solutions.face_detection
    mp_face_mesh = mp.solutions.face_mesh
    mp_drawing = mp.solutions.drawing_utils
    
    # Create detection models
    face_detection = mp_face_detection.FaceDetection(
        model_selection=0, 
        min_detection_confidence=0.5
    )
    
    face_mesh = mp_face_mesh.FaceMesh(
        static_image_mode=True, 
        max_num_faces=1, 
        min_detection_confidence=0.5
    )
    
    print("✓ MediaPipe face detection initialized successfully!")
    MEDIAPIPE_AVAILABLE = True
    
    # Test with a simple synthetic image
    test_image = np.ones((200, 200, 3), dtype=np.uint8) * 128
    cv2.circle(test_image, (70, 80), 5, (0, 0, 0), -1)  # Left eye
    cv2.circle(test_image, (130, 80), 5, (0, 0, 0), -1)  # Right eye
    cv2.rectangle(test_image, (95, 110), (105, 120), (0, 0, 0), -1)  # Nose
    cv2.rectangle(test_image, (80, 140), (120, 150), (0, 0, 0), -1)  # Mouth
    
    rgb_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
    results = face_detection.process(rgb_image)
    
    if results.detections:
        print(f"✓ Face detection test successful - found {len(results.detections)} face(s)")
    else:
        print("✓ Face detection test complete - synthetic face not detected (normal)")
    
except Exception as e:
    print(f"❌ MediaPipe face detection failed: {e}")
    MEDIAPIPE_AVAILABLE = False

Initializing MediaPipe face detection...
✓ PyTorch loaded
✓ MediaPipe face detection initialized successfully!
✓ Face detection test complete - synthetic face not detected (normal)


In [17]:
def ensure_openface_ready() -> bool:
    """Ensure OpenFace resources are present and neural net + aligner are instantiated."""
    if not OPENFACE_AVAILABLE:
        return False
    global openface_net, openface_aligner
    if openface_net is not None and openface_aligner is not None:
        return True
    try:
        import urllib.request
        import bz2
        import shutil
        OPENFACE_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
        if not OPENFACE_MODEL_PATH.exists():
            print("Downloading OpenFace model (nn4.small2.v1.t7)...")
            urllib.request.urlretrieve(OPENFACE_MODEL_URL, str(OPENFACE_MODEL_PATH))
        OPENFACE_LANDMARK_PATH.parent.mkdir(parents=True, exist_ok=True)
        if not OPENFACE_LANDMARK_PATH.exists():
            compressed_path = OPENFACE_LANDMARK_PATH.with_suffix(OPENFACE_LANDMARK_PATH.suffix + ".bz2")
            print("Downloading OpenFace landmark predictor (68 points)...")
            urllib.request.urlretrieve(OPENFACE_LANDMARK_URL, str(compressed_path))
            with bz2.open(compressed_path, "rb") as src, open(OPENFACE_LANDMARK_PATH, "wb") as dst:
                shutil.copyfileobj(src, dst)
            compressed_path.unlink(missing_ok=True)
        openface_net = openface.TorchNeuralNet(str(OPENFACE_MODEL_PATH), imgDim=96, cuda=False)
        openface_aligner = openface.AlignDlib(str(OPENFACE_LANDMARK_PATH))
        return True
    except Exception as exc:
        print(f"OpenFace initialization failed: {exc}")
        openface_net = None
        openface_aligner = None
        return False

In [5]:
try:
    import face_recognition  # dlib-based 128-d embeddings
    FACE_RECOGNITION_AVAILABLE = True
    print("face_recognition backend available ✓")
except Exception as e:
    FACE_RECOGNITION_AVAILABLE = False
    print(f"face_recognition backend unavailable: {e}")

C:\Users\Kailu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\face_recognition_models\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


face_recognition backend available ✓


In [18]:
try:
    import openface
    OPENFACE_AVAILABLE = True
    print("openface backend available ✓")
except Exception as e:
    OPENFACE_AVAILABLE = False
    print("openface backend unavailable:", e)
    print("Install with `pip install openface torchfile` and ensure dependencies are met.")

openface backend unavailable: No module named 'openface'
Install with `pip install openface torchfile` and ensure dependencies are met.


## 3. Build Face Database

In [6]:
# Helpers to load images and combine embeddings
import os, glob
from typing import Dict, List
import torch

def collect_images_for_people(person_dirs: Dict[str, str]) -> Dict[str, List[str]]:
    """Return a sorted list of image paths for each person."""
    exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp")
    images_by_person: Dict[str, List[str]] = {}
    for person, pdir in person_dirs.items():
        files: List[str] = []
        for ext in exts:
            files.extend(glob.glob(os.path.join(pdir, ext)))
        files.sort()
        images_by_person[person] = files
    return images_by_person

def mean_embedding(embeddings: List[torch.Tensor]) -> torch.Tensor | None:
    """Compute a unit-normalized mean embedding from a list of tensors."""
    if not embeddings:
        return None
    stack = torch.stack(embeddings, dim=0)
    mean_vec = stack.mean(dim=0)
    mean_vec = mean_vec / (torch.norm(mean_vec) + 1e-8)
    return mean_vec

In [7]:
print("Building face dataset for KNN training...")

# Prepare containers
avg_embeddings: Dict[str, torch.Tensor | None] = {}
embeddings: List[np.ndarray] = []
labels: List[str] = []

# Discover all available identities under facebank/facebank (excluding test_images)
base_root = os.path.join("facebank", "facebank")
skip_names = {"test_images"}
skip_names_lower = {name.lower() for name in skip_names}
people_dirs: Dict[str, str] = {}

if not os.path.isdir(base_root):
    raise RuntimeError(f"Base facebank directory not found at {base_root}")

for entry in sorted(os.listdir(base_root), key=str.lower):
    full_path = os.path.join(base_root, entry)
    if not os.path.isdir(full_path):
        continue
    if entry.lower() in skip_names_lower:
        continue
    people_dirs[entry] = full_path

if not people_dirs:
    raise RuntimeError("No person directories discovered under facebank; please check the dataset structure.")
print(f"Discovered {len(people_dirs)} identities in {base_root}")

# Collect images per person
images_by_person = collect_images_for_people(people_dirs)

total_images = 0
for person, imgs in images_by_person.items():
    print(f"{person}: found {len(imgs)} image(s)")
    total_images += len(imgs)
print(f"Total images located: {total_images}")

# Build embedding matrix and per-person averages
for person, img_list in images_by_person.items():
    print(f"\nProcessing {person}...")
    person_embeddings: List[torch.Tensor] = []
    for image_path in img_list:
        emb = get_embeddings_openface(image_path)
        if emb is None:
            print(f"  Skipping (no embedding): {image_path}")
            continue
        person_embeddings.append(emb)
        embeddings.append(emb.numpy())
        labels.append(person)
    avg_embeddings[person] = mean_embedding(person_embeddings)
    print(f"  Stored {len(person_embeddings)} embeddings")
    if avg_embeddings[person] is not None:
        print(f"  Mean embedding norm: {torch.norm(avg_embeddings[person]):.4f}")
    else:
        print("  No valid embeddings to average")

Building face dataset for KNN training...
Discovered 31 identities in facebank\facebank
Akshay Kumar: found 41 image(s)
Alexandra Daddario: found 41 image(s)
Alia Bhatt: found 41 image(s)
Amitabh Bachchan: found 41 image(s)
Andy Samberg: found 41 image(s)
Anushka Sharma: found 41 image(s)
Billie Eilish: found 41 image(s)
Brad Pitt: found 41 image(s)
Camila Cabello: found 41 image(s)
Charlize Theron: found 41 image(s)
Claire Holt: found 41 image(s)
Courtney Cox: found 41 image(s)
Dwayne Johnson: found 41 image(s)
Elizabeth Olsen: found 41 image(s)
Ellen Degeneres: found 41 image(s)
Henry Cavill: found 41 image(s)
Hrithik Roshan: found 41 image(s)
Hugh Jackman: found 41 image(s)
Jessica Alba: found 41 image(s)
Kashyap: found 30 image(s)
Lisa Kudrow: found 41 image(s)
Margot Robbie: found 41 image(s)
Marmik: found 32 image(s)
Natalie Portman: found 41 image(s)
Priyanka Chopra: found 41 image(s)
Robert Downey Jr: found 41 image(s)
Roger Federer: found 41 image(s)
Tom Cruise: found 41 image

In [ ]:
if not embeddings:
    raise RuntimeError("No embeddings were generated; please check the data paths.")

embedding_matrix = np.vstack(embeddings)
label_array = np.array(labels)

from collections import Counter
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score,
    top_k_accuracy_score,
 )
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd

class_counts = Counter(labels)
min_class_size = min(class_counts.values())
n_splits = max(2, min(5, min_class_size))
print(f"Using {n_splits}-fold stratified CV (smallest class has {min_class_size} samples)")

candidate_k = [1, 3, 5, 7, 9]
weighting_options = ["uniform", "distance"]
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
cv_rows = []
for k in candidate_k:
    for weight in weighting_options:
        model = KNeighborsClassifier(n_neighbors=k, metric="euclidean", weights=weight)
        scores = cross_val_score(model, embedding_matrix, label_array, cv=cv)
        cv_rows.append({
            "n_neighbors": k,
            "weights": weight,
            "mean_accuracy": scores.mean(),
            "std_accuracy": scores.std(),
        })
cv_results = pd.DataFrame(cv_rows).sort_values("mean_accuracy", ascending=False)
display(cv_results)

best_row = cv_results.iloc[0]
best_k = int(best_row["n_neighbors"])
best_weight = str(best_row["weights"])
print(f"Best CV config: k={best_k}, weights={best_weight} (mean acc {best_row['mean_accuracy']:.4f})")

X_train, X_val, y_train, y_val = train_test_split(
    embedding_matrix, label_array, test_size=0.2, stratify=label_array, random_state=42
 )
eval_model = KNeighborsClassifier(n_neighbors=best_k, metric="euclidean", weights=best_weight)
eval_model.fit(X_train, y_train)
y_val_pred = eval_model.predict(X_val)
print("\nValidation classification report:")
print(classification_report(y_val, y_val_pred, digits=3))

val_accuracy = accuracy_score(y_val, y_val_pred)
val_balanced_acc = balanced_accuracy_score(y_val, y_val_pred)
val_macro_f1 = f1_score(y_val, y_val_pred, average="macro", zero_division=0)
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Validation balanced accuracy: {val_balanced_acc:.4f}")
print(f"Validation macro F1: {val_macro_f1:.4f}")

if len(np.unique(y_val)) > 1:
    try:
        val_top3 = top_k_accuracy_score(y_val, eval_model.predict_proba(X_val), k=min(3, len(np.unique(y_val))))
        print(f"Validation top-{min(3, len(np.unique(y_val)))} accuracy: {val_top3:.4f}")
    except AttributeError:
        print("Top-k accuracy unavailable (predict_proba not supported).")

val_distances, _ = eval_model.kneighbors(X_val, n_neighbors=1)
validation_threshold = float(val_distances.mean() + 2 * val_distances.std())
print(f"Suggested distance threshold from validation: {validation_threshold:.4f}")

labels_sorted = np.unique(label_array)
cm = confusion_matrix(y_val, y_val_pred, labels=labels_sorted)
cm_df = pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted)
print("\nValidation confusion matrix (rows=true, cols=pred):")
display(cm_df)
try:
    import seaborn as sns
    import matplotlib.pyplot as plt

    plt.figure(figsize=(min(14, 1.2 * len(labels_sorted)), min(10, 0.8 * len(labels_sorted))))
    sns.heatmap(cm_df, fmt="d", annot=len(labels_sorted) <= 30, cmap="Blues")
    plt.title("Validation Confusion Matrix")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Seaborn not installed; skipping heatmap.")

knn = KNeighborsClassifier(n_neighbors=best_k, metric="euclidean", weights=best_weight)
knn.fit(embedding_matrix, label_array)
train_distances, _ = knn.kneighbors(embedding_matrix, n_neighbors=1)
distance_threshold = float(train_distances.mean() + 2 * train_distances.std())
print(f"\nFinal KNN trained on {embedding_matrix.shape[0]} samples")
print(f"Final distance threshold (mean + 2σ): {distance_threshold:.4f}")

Using 5-fold stratified CV (smallest class has 30 samples)


,n_neighbors,weights,mean_accuracy,std_accuracy
2,3,uniform,0.993613,0.005403
0,1,uniform,0.992813,0.005285
1,1,distance,0.992813,0.005285
3,3,distance,0.992813,0.005285
7,7,distance,0.992810,0.002978
6,7,uniform,0.992013,0.004359
5,5,distance,0.992010,0.002515
9,9,distance,0.992010,0.002515
4,5,uniform,0.991213,0.003896
8,9,uniform,0.991213,0.003896


Best CV config: k=3, weights=uniform (mean acc 0.9936)

Validation classification report:
                    precision    recall  f1-score   support

      Akshay Kumar      1.000     1.000     1.000         8
Alexandra Daddario      1.000     1.000     1.000         8
        Alia Bhatt      1.000     1.000     1.000         8
  Amitabh Bachchan      1.000     1.000     1.000         8
      Andy Samberg      1.000     1.000     1.000         8
    Anushka Sharma      1.000     1.000     1.000         8
     Billie Eilish      0.900     1.000     0.947         9
         Brad Pitt      1.000     1.000     1.000         9
    Camila Cabello      1.000     0.875     0.933         8
   Charlize Theron      1.000     1.000     1.000         8
       Claire Holt      1.000     1.000     1.000         9
      Courtney Cox      1.000     1.000     1.000         8
    Dwayne Johnson      1.000     1.000     1.000         8
   Elizabeth Olsen      1.000     1.000     1.000         8
   Ellen 

In [11]:
import joblib
from pathlib import Path

def _serialize_avg_embeddings(avg_embeds: dict[str, torch.Tensor | None]) -> dict[str, np.ndarray | None]:
    serialized: dict[str, np.ndarray | None] = {}
    for name, tensor in avg_embeds.items():
        serialized[name] = tensor.numpy() if tensor is not None else None
    return serialized

def save_facebank(payload_path: str = "artifacts/facebank_knn.pkl") -> None:
    """Persist the trained KNN, labels, thresholds, and mean embeddings to disk."""
    if "knn" not in globals() or "label_array" not in globals():
        raise RuntimeError("KNN classifier has not been trained yet; run the training cell first.")
    package = {
        "knn": knn,
        "label_array": label_array,
        "distance_threshold": distance_threshold,
        "validation_threshold": validation_threshold,
        "avg_embeddings": _serialize_avg_embeddings(avg_embeddings),
        "embedding_matrix": embedding_matrix,
        "people_dirs": people_dirs,
    }
    path = Path(payload_path)
    if path.parent and not path.parent.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(package, path)
    print(f"[INFO] Saved facebank artefacts to {path.resolve()}")

def load_facebank(payload_path: str = "artifacts/facebank_knn.pkl") -> dict:
    """Load persisted artefacts; returns a dictionary with the stored objects."""
    path = Path(payload_path)
    if not path.exists():
        raise FileNotFoundError(f"Facebank file not found at {path.resolve()}")
    package = joblib.load(path)
    print(f"[INFO] Loaded facebank artefacts from {path.resolve()}")
    return package

# Example usage:
# save_facebank()
# restored = load_facebank()
# restored_knn = restored["knn"]

In [15]:
import os
import glob

if "base_root" not in globals():
    base_root = os.path.join("facebank", "facebank")
    print(f"base_root not preset; defaulting to {base_root}")

# Configure the target directory and filename you'd like to test
person_name = "test_images"  # change to a known folder or "test_images" for unseen photos
desired_filename = "Courtney Cox_66.jpg"  # set to the exact file name you want to try
manual_path = None  # set to a full explicit path if you want to bypass directory lookup

search_roots = []
primary_dir = os.path.join(base_root, person_name)
alternate_dir = os.path.join(base_root, person_name.replace(" ", "_"))
test_images_dir = os.path.join(base_root, "test_images")

for candidate_dir in {primary_dir, alternate_dir, test_images_dir}:
    if os.path.isdir(candidate_dir):
        search_roots.append(candidate_dir)

if manual_path and os.path.exists(manual_path):
    test_image = manual_path
else:
    found = False
    for root in search_roots:
        candidate_path = os.path.join(root, desired_filename)
        if os.path.exists(candidate_path):
            test_image = candidate_path
            found = True
            break
    if not found:
        if not search_roots:
            raise FileNotFoundError(
                f"No valid directories found for testing. Checked {primary_dir}, {alternate_dir}, {test_images_dir}."
            )
        print(
            f"Requested image '{desired_filename}' not located. Listing candidates in {len(search_roots)} folder(s)."
        )
        available_images = []
        for root in search_roots:
            available_images.extend(glob.glob(os.path.join(root, "*.jpg")))
            available_images.extend(glob.glob(os.path.join(root, "*.jpeg")))
            available_images.extend(glob.glob(os.path.join(root, "*.png")))
        if not available_images:
            raise FileNotFoundError(
                f"No images found under {[os.path.basename(root) for root in search_roots]}"
            )
        available_images.sort()
        test_image = available_images[0]
        print(f"Using fallback image: {os.path.basename(test_image)} from {os.path.dirname(test_image)}")

test_embedding = get_embeddings_openface(test_image)
if test_embedding is None:
    raise RuntimeError(f"Unable to extract embedding for {test_image}")

query = test_embedding.numpy()
neighbor_distances, neighbor_indices = knn.kneighbors([query], n_neighbors=min(5, len(label_array)))
nearest_distance = float(neighbor_distances[0][0])
predicted_label = knn.predict([query])[0]

print(f"Test image: {test_image}")
print(f"Predicted label: {predicted_label}")
print(f"Nearest neighbor distance: {nearest_distance:.4f} (threshold {distance_threshold:.4f})")
if nearest_distance < distance_threshold:
    print("Prediction flagged as UNKNOWN (distance above threshold)")

print("Top neighbor candidates:")
for rank, (idx, dist) in enumerate(zip(neighbor_indices[0], neighbor_distances[0]), start=1):
    print(f"  {rank}. {label_array[idx]} @ distance {dist:.4f}")

Test image: facebank\facebank\test_images\Courtney Cox_66.jpg
Predicted label: Courtney Cox
Nearest neighbor distance: 0.1994 (threshold 0.0000)
Top neighbor candidates:
  1. Courtney Cox @ distance 0.1994
  2. Courtney Cox @ distance 0.2053
  3. Courtney Cox @ distance 0.2054
  4. Courtney Cox @ distance 0.2209
  5. Courtney Cox @ distance 0.2219
